# Урок 2.6. Расширенные стратегии: semantic chunking, parent-document, cross-encoder

**Модуль 2 «Embeddings и векторные базы данных» · Курс «LLM для Python-разработчиков»**

В уроке 2.5 вы собрали базовый арсенал: чанкинг по структуре, метаданные, диверсификация. Сегодня — три приёма следующего уровня и один почти бесплатный трюк. Как обычно, мы не поверим ни одному из них на слово: что можно измерить — измерим, а один приём даже честно провалим лексикой, чтобы понять, зачем ему эмбеддинги.

## 🎯 Цели урока

После этого урока вы сможете:

- добавлять **контекстные заголовки** к чанкам — самый дешёвый апгрейд индекса;
- реализовать **semantic chunking** (границы по смыслу) и объяснить, почему это единственный приём модуля, которому по-настоящему нужны эмбеддинги;
- строить **parent-document retrieval**: искать мелким, отдавать крупное;
- объяснить разницу **bi-encoder vs cross-encoder** и посчитать, почему cross-encoder живёт только на второй ступени;
- расставить все приёмы модуля по карте «эффект ↔ цена».

## План

1. Контекстные заголовки
2. Semantic chunking: акт 1 — провал лексики, акт 2 — эмбеддинги
3. Parent-document retrieval
4. Cross-encoder: реранкер золотого стандарта
5. Итоговая карта приёмов

⏱ **Время:** ~75–90 минут.

> 💡 Живо без скачиваний: заголовки, алгоритм semantic chunking (с лексикой), parent-document, арифметика cross-encoder'а. Модели (e5 из урока 2.1 и cross-encoder) нужны для двух guarded-секций.

## 1. Контекстные заголовки: почти бесплатный апгрейд

Взгляните на типичный «глубокий» чанк из середины документа:

> «Компенсация за ночные вызовы — отгул на следующий день.»

Вырванный из документа, он потерял главное — **контекст**: чьи вызовы? какой процесс? Человек, читающий подряд, помнит заголовок «Инциденты и дежурства»; вектор чанка — нет. А вопросы пользователей часто содержат слова именно из заголовка («дежурство»), а не из глубокого абзаца.

Приём — **contextual chunk headers**: при индексации приклеить к каждому чанку заголовок документа (а в длинных доках — и заголовок раздела):

```text
текст для индексации = "Инциденты и дежурства. Компенсация за ночные вызовы - ..."
текст для показа     = оригинальный чанк (храним отдельно, в payload!)
```

Проверим на нашем поиске из урока 2.5 (TF-IDF-лайт переносим сюда — урок самодостаточен):

In [ ]:
import math
import re

from utils import chunk_paragraphs, load_documents, split_sentences

docs = load_documents("data")

def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())

def build_lexical_index(texts: list[str]):
    token_lists = [tokenize(t) for t in texts]
    df: dict[str, int] = {}
    for tokens in token_lists:
        for word in set(tokens):
            df[word] = df.get(word, 0) + 1
    idf = {word: math.log(len(texts) / count) for word, count in df.items()}
    return token_lists, idf

def lexical_search(query: str, token_lists, idf, top_k: int = 5):
    scored = []
    for i, tokens in enumerate(token_lists):
        score = sum((tokens.count(word) / len(tokens)) * idf.get(word, 0.0)
                    for word in set(tokenize(query)))
        scored.append((score, i))
    return sorted(scored, reverse=True)[:top_k]

def make_chunks(with_header: bool):
    """[(doc_id, chunk_index, текст_для_индексации)] по всем документам."""
    items = []
    for doc in docs:
        for i, piece in enumerate(chunk_paragraphs(doc.text, max_chars=500)):
            # заголовок уже есть в чанке 0 (это первая строка документа) -
            # приклеиваем только глубоким чанкам
            text = f"{doc.title}. {piece}" if (with_header and i > 0) else piece
            items.append((doc.doc_id, i, text))
    return items

question = "Какая надбавка за неделю дежурства?"
for with_header in (False, True):
    items = make_chunks(with_header)
    token_lists, idf = build_lexical_index([t for _, _, t in items])
    (score, i), *_ = lexical_search(question, token_lists, idf)
    doc_id, chunk_index, _ = items[i]
    label = "с заголовками:" if with_header else "без заголовков:"
    print(f"{label:<16} нашли {doc_id}_{chunk_index}, score={score:.3f}")

Честный разбор: на нашей мини-базе **ранг не изменился** (правильный чанк и так был первым — авторы «Векторики» пишут связно, и слово «дежурство» есть в каждом абзаце), но **score вырос** — заголовок добавил совпадений. Этот запас прочности и есть смысл приёма: на реальных данных, где глубокие абзацы начинаются с «Он также...» и «Это значит...», заголовок часто решает, найдётся фрагмент вообще или нет.

Правила приёма:

- **индексируем** текст с заголовком, **показываем** (и храним в payload) оригинал;
- цена — пара десятков токенов на чанк при индексации, в рантайме — ноль;
- вывод: **делайте по умолчанию**, выключайте только если eval покажет вред (не покажет).

## 2. Semantic chunking: границы по смыслу

Абзацная нарезка (урок 2.5) опирается на структуру от автора. А если структуры нет? Расшифровка звонка, OCR-скан, выгрузка из чата — сплошная «каша» предложений. **Semantic chunking** ищет границы сам:

```text
1. Режем текст на предложения.
2. Для каждой границы меряем похожесть "окна до" и "окна после".
3. Кривая похожести: внутри темы - высокая, на смене темы - ДОЛИНА.
4. Режем в долинах (порог - например, нижний перцентиль кривой).

похожесть
   ^
   |  ####    ####      ####  ####
   |  ####    ####  ##  ####  ####
   |  ####  # ####  ##  ####  ####
   +--------^-----------^------------> границы предложений
           долина      долина
           РЕЗАТЬ      РЕЗАТЬ
```

Алгоритм не привязан к тому, ЧЕМ мерить похожесть, — напишем его с подключаемой функцией и сначала попробуем нашу лексическую (Jaccard по стеммам). Полигон — «каша» из трёх склеенных документов без заголовков и абзацев; настоящие границы тем мы знаем заранее:

In [ ]:
import numpy as np

def sentence_stems(sentence: str) -> set[str]:
    """Грубые "стеммы": первые 5 букв слова (отпуск/отпуска -> 'отпус')."""
    return {w[:5] for w in tokenize(sentence) if len(w) > 2}

def lexical_window_sim(left: list[str], right: list[str]) -> float:
    """Похожесть двух наборов предложений по пересечению стеммов (Jaccard)."""
    a = set().union(*map(sentence_stems, left)) if left else set()
    b = set().union(*map(sentence_stems, right)) if right else set()
    return len(a & b) / len(a | b) if a | b else 0.0

def similarity_curve(sentences: list[str], sim_fn, window: int = 2) -> list[float]:
    """Похожесть "окна до" и "окна после" для каждой границы предложений."""
    return [
        sim_fn(sentences[max(0, i - window):i], sentences[i:i + window])
        for i in range(1, len(sentences))
    ]

# полигон: три документа, склеенные в кашу без заголовков и абзацев
def body(doc_id: str) -> str:
    text = next(d for d in docs if d.doc_id == doc_id).text
    return " ".join(text.splitlines()[1:]).strip()

parts = [body("otpusk"), body("vpn"), body("code-review")]
sentences = split_sentences(" ".join(parts))
true_boundaries = {
    len(split_sentences(parts[0])),
    len(split_sentences(parts[0])) + len(split_sentences(parts[1])),
}
print(f"Каша: {len(sentences)} предложений, настоящие границы тем "
      f"перед предложениями {sorted(true_boundaries)}\n")

curve = similarity_curve(sentences, lexical_window_sim)
for i, sim in enumerate(curve, start=1):
    bar = "#" * int(sim * 50)
    mark = "  <-- настоящая граница тем" if i in true_boundaries else ""
    print(f"{i:3} {sim:.3f} {bar}{mark}")

Посмотрите на эту кривую: она — **шум**. Нулевые «долины» рассыпаны по всему тексту (ложные границы!), а настоящие границы тем ничем не выделяются. Порог здесь ставить некуда.

Почему так? Соседние предложения одной темы часто **не имеют общих слов вообще**: «Заявку на отпуск нужно подать за 14 дней» и «Неиспользованные дни переносятся» — об одном, но лексически чужие. Это «пёс ≠ собака» из урока 2.1, возведённое в степень: на коротких текстах пересечение слов — слишком редкое событие, чтобы быть сигналом.

Отсюда главный вывод секции: **semantic chunking — единственный приём модуля, которому эмбеддинги нужны по-настоящему.** Чанкинг по структуре, заголовки, parent-document, диверсификация — всё это работало без модели. Здесь без неё не работает ничего: нужна похожесть, которая видит смысл сквозь разные слова, — то есть косинус векторов e5.

Акт 2 — для владельцев модели (урок 2.1). Тот же алгоритм, меняется только `sim_fn`:

In [ ]:
EMB_MODEL_NAME = "intfloat/multilingual-e5-small"

model = None
try:
    from sentence_transformers import SentenceTransformer
    from huggingface_hub import snapshot_download
    snapshot_download(EMB_MODEL_NAME, local_files_only=True)
    model = SentenceTransformer(EMB_MODEL_NAME)
    print("Модель e5 загружена — считаем смысловую кривую.")
except Exception:
    print("Модель e5 не в кэше (урок 2.1) — акт 2 пропустится.")

if model is not None:
    # эмбеддим каждое предложение ОДИН раз, окна складываем из готовых векторов
    sent_vecs = model.encode([f"query: {s}" for s in sentences],
                             normalize_embeddings=True)

    def window_vec(indices) -> np.ndarray:
        v = sent_vecs[list(indices)].mean(axis=0)
        return v / np.linalg.norm(v)

    W = 2
    sem_curve = [
        float(window_vec(range(max(0, i - W), i)) @
              window_vec(range(i, min(len(sentences), i + W))))
        for i in range(1, len(sentences))
    ]

    threshold = float(np.percentile(sem_curve, 10))  # нижние 10% - кандидаты в границы
    print(f"Порог (10-й перцентиль кривой): {threshold:.3f}\n")
    for i, sim in enumerate(sem_curve, start=1):
        bar = "#" * int(max(sim, 0) * 40)
        cut = " РЕЗАТЬ" if sim <= threshold else ""
        mark = "  <-- настоящая граница" if i in true_boundaries else ""
        print(f"{i:3} {sim:.3f} {bar}{cut}{mark}")

Ожидаемая картина (проверьте на своей машине): внутри темы смысловая похожесть высокая — эмбеддинги видят, что «заявка на отпуск» и «неиспользованные дни» из одного мира, — а на смене темы кривая ныряет. Нижний перцентиль как порог находит настоящие границы, почти не создавая ложных.

Практические ручки semantic chunking:

- **окно** (у нас 2 предложения): шире — глаже кривая, но границы «размазываются»;
- **порог**: перцентиль кривой (10–25%) надёжнее абсолютного значения — он адаптируется к тексту;
- **пост-обработка**: склеить чанки короче минимума, дорезать длиннее максимума (лимит эмбеддера из урока 2.5 никуда не делся!);
- **цена**: эмбеддинг каждого предложения — на порядок больше вызовов модели, чем при абзацной нарезке.

Итого: если у текста есть структура — она бесплатна и обычно не хуже. Semantic chunking — инструмент для «каши», а не замена абзацам.

## 3. Parent-document retrieval: искать мелко, отдавать крупно

В упражнении урока 2.5 вы подтягивали соседние чанки. Обобщение этой идеи — **развести две роли текста**, которые мы до сих пор смешивали:

```text
             для ПОИСКА                    для ОТВЕТА
             мелкие чанки (~200-300)       родитель: документ или крупный чанк
             точный вектор                 связный контекст
                    |                             ^
                    |     child.parent_id         |
                    +-----------------------------+

индекс: мелкие чанки -> каждый знает parent_id
поиск:  топ-k мелких -> дедупликация родителей -> отдаём родителей
```

Мелкий чанк — снайпер (урок 2.5: маленьким чанкам — точное попадание), родитель — полная картина для человека или LLM. Реализация — на нашем лексическом поиске, целиком живая:

In [ ]:
# child-чанки: мелкие (до 400 символов - примерно абзац), каждый знает родителя.
# Не мельчим сильнее: у TF-IDF есть перекос к коротким текстам (одно совпадение
# в крошечном чанке даёт огромный tf) - урок 2.5 про крайности никуда не делся
children = []
for doc in docs:
    for i, piece in enumerate(chunk_paragraphs(doc.text, max_chars=400)):
        children.append({"parent_id": doc.doc_id, "chunk_index": i, "text": piece})

parents = {doc.doc_id: doc for doc in docs}
child_tokens, child_idf = build_lexical_index([c["text"] for c in children])
print(f"Индекс: {len(children)} мелких чанков -> {len(parents)} родителей\n")

def parent_search(question: str, top_k_children: int = 6, top_parents: int = 2):
    """Ищем мелкими чанками, отдаём родительские документы (с дедупликацией)."""
    hits = lexical_search(question, child_tokens, child_idf, top_k=top_k_children)
    seen, results = set(), []
    for score, i in hits:
        child = children[i]
        if child["parent_id"] in seen:
            continue                      # родитель уже в выдаче - пропускаем дубль
        seen.add(child["parent_id"])
        results.append((score, child))
        if len(results) == top_parents:
            break
    return results

for question in ("Сколько компания доплачивает в первые дни болезни?",
                 "Какая надбавка за неделю дежурства?"):
    print(f"Вопрос: {question}")
    for score, child in parent_search(question):
        parent = parents[child["parent_id"]]
        print(f"  попали чанком {child['parent_id']}_{child['chunk_index']} "
              f"({len(child['text'])} симв., score={score:.3f})")
        print(f"  отдаём родителя «{parent.title}» ({len(parent.text)} симв.)")
    print()

Разделение ролей чисто видно в выводе: **попадание** — фрагментом в ~200 символов (точный вектор — помните из 2.5, как мелкие чанки концентрируют score), **выдача** — целым документом со всем контекстом. И заметьте дедупликацию: три чанка одного документа не превращаются в трёх одинаковых родителей.

Компромисс очевиден: родитель стоит дороже в токенах промпта (модуль 3 напомнит об этом счётом из урока 1.3). Поэтому в проде «родителем» часто назначают не документ, а **крупный чанк** (800–1500 символов) или окно вокруг попадания — упражнение 2 как раз про это. Правило выбора: чем длиннее исходные документы, тем важнее ограничивать родителя.

## 4. Cross-encoder: реранкер золотого стандарта

Всё это время наши модели считали вектор запроса и вектор документа **независимо** — это называется **bi-encoder** (две «башни»):

```text
BI-ENCODER (e5):                       CROSS-ENCODER:

запрос -> [модель] -> вектор q          запрос + документ
документ -> [модель] -> вектор d          вместе, одной строкой
                                            |
score = q @ d  (дёшево!)                [модель читает пару]
векторы d предвычислены                     |
и лежат в индексе                       score (одно число)
```

Cross-encoder читает запрос и документ **вместе**: attention (урок 1.1!) сопоставляет каждое слово вопроса с каждым словом текста. Он видит связи, которые теряются при сжатии в два независимых вектора, — поэтому стабильно точнее на переранжировании. Но у этой точности структурная цена: **у cross-encoder'а нет вектора документа**. Нечего предвычислить, нечего сложить в индекс — каждая пара «запрос-документ» требует полного прохода модели в рантайме. Посчитаем, что это значит:

In [ ]:
CE_MS_PER_PAIR = 25       # проход cross-encoder по паре на CPU, ~мс (порядок)
BI_SEARCH_MS = 15         # bi-encoder: эмбеддинг запроса + поиск по HNSW, ~мс

print(f"{'сценарий':<42} {'время на 1 запрос':>18}")
print("-" * 62)

for n_docs in (10_000, 1_000_000):
    full_ce = n_docs * CE_MS_PER_PAIR / 1000
    label = f"cross-encoder по ВСЕЙ базе ({n_docs:,})"
    time_str = f"{full_ce / 3600:.1f} часа(ов)" if full_ce > 3600 else f"{full_ce:,.0f} с"
    print(f"{label:<42} {time_str:>18}")

for top_n in (10, 30, 100):
    two_stage = BI_SEARCH_MS + top_n * CE_MS_PER_PAIR
    label = f"двухступенчато: bi топ-{top_n} -> CE"
    print(f"{label:<42} {two_stage / 1000:>16.2f} с")

Вот и вся архитектура современного поиска в одной таблице: cross-encoder по базе — часы, cross-encoder по 30 кандидатам от bi-encoder'а — меньше секунды. **Первая ступень отвечает за «не потерять» (recall), вторая — за «поставить лучшее наверх» (precision)** — схема из урока 2.5, теперь с настоящей моделью на второй ступени.

Для русского языка бесплатный вариант — мультиязычный **`cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`** (~500 МБ с зависимостями, скачается с huggingface.co при первом использовании). Желающие могут прогнать живое демо:

In [ ]:
CE_MODEL_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

ce_model = None
try:
    from sentence_transformers import CrossEncoder
    from huggingface_hub import snapshot_download
    snapshot_download(CE_MODEL_NAME, local_files_only=True)
    ce_model = CrossEncoder(CE_MODEL_NAME)
    print("Cross-encoder загружен.")
except Exception:
    print(f"Cross-encoder не в кэше — для демо скачайте его:")
    print(f"  CrossEncoder('{CE_MODEL_NAME}')  # один раз, ~500 МБ")

if model is not None and ce_model is not None:
    # ступень 1: e5 приносит кандидатов по чанкам «Векторики»
    all_chunks = []
    for doc in docs:
        for piece in chunk_paragraphs(doc.text, max_chars=500):
            all_chunks.append((doc.doc_id, piece))
    chunk_vecs = model.encode([f"passage: {t}" for _, t in all_chunks],
                              normalize_embeddings=True)

    question = "Упал прод, что делать в первую очередь?"
    q_vec = model.encode([f"query: {question}"], normalize_embeddings=True)[0]
    candidate_ids = np.argsort(chunk_vecs @ q_vec)[::-1][:8]

    # ступень 2: cross-encoder читает пары (вопрос, чанк) целиком
    pairs = [(question, all_chunks[i][1]) for i in candidate_ids]
    ce_scores = ce_model.predict(pairs)

    print(f"Вопрос: {question}\n")
    print(f"{'место':>5} {'bi-encoder (e5)':<22} {'после cross-encoder':<22}")
    reranked = [candidate_ids[j] for j in np.argsort(ce_scores)[::-1]]
    for rank in range(5):
        bi_id = all_chunks[candidate_ids[rank]][0]
        ce_id = all_chunks[reranked[rank]][0]
        print(f"{rank + 1:>5} {bi_id:<22} {ce_id:<22}")

Когда включать cross-encoder:

- **высокая цена ошибки топа**: поддержка, юридические и медицинские базы, поиск по регламентам;
- кандидатов на второй ступени немного (20–50) — CPU справляется;
- eval показывает, что bi-encoder путает близкие документы (частый случай: несколько документов об одном, отличаются деталями).

Альтернатива с другого конца спектра — **LLM-as-reranker**: попросить LLM оценить релевантность кандидатов (промпт + JSON-оценки, всё из модуля 1). Дороже и медленнее cross-encoder'а, зато без новой модели и с объяснениями. Встретимся с ним в модуле 3; заготовку промпта вы напишете в упражнении 3 уже сейчас.

## 5. Итоговая карта приёмов

Синтез уроков 2.5–2.6 — вешайте рядом с чек-листом чанкинга:

| приём | что улучшает | цена | когда включать |
|---|---|---|---|
| чанкинг по структуре | точность фрагмента | ~0 | всегда (2.5) |
| перекрытие окна | потери на границах | +15–25% индекса | тексты без структуры (2.5) |
| контекстные заголовки | находимость глубоких чанков | ~0 | по умолчанию |
| метаданные + фильтры | бизнес-правила поиска | ~0 | всегда (2.3–2.5) |
| диверсификация | разнообразие выдачи | риск (тёмная сторона, 2.5) | многочанковые документы |
| parent-document | связность контекста | токены промпта | RAG с мелкими чанками |
| semantic chunking | границы в «каше» | эмбеддинг каждого предложения | нет структуры |
| cross-encoder | точность топ-3 | латентность 2-й ступени | высокая цена ошибки |

Порядок внедрения — сверху вниз: сначала бесплатное, потом дорогое, и **каждый шаг — через eval** (урок 1.7 навсегда с нами).

## ⚠️ Частые ошибки

1. **Semantic chunking на текстах с готовой структурой.** Платить эмбеддингом каждого предложения за то, что автор уже разметил абзацами, — расточительство. Сначала структура, «каша» — потом.
2. **Cross-encoder как первая ступень.** У него нет предвычислимых векторов — только полный проход по каждой паре. Часы на миллионе документов. Только вторая ступень.
3. **Англоязычный реранкер на русских текстах.** `ms-marco-MiniLM` без приставки `mmarco` — английский; на русском он тихо выдаст шум. Знакомая песня из урока 2.1 — читайте карточку модели.
4. **Родитель без лимита.** Parent-document на 20-страничном документе отправит в промпт LLM всю главу. Ограничивайте родителя размером или окном (упражнение 2).
5. **Индексировать один текст, показывать его же.** Правильно: индексировать обогащённый (заголовок + чанк), показывать оригинал из payload. Смешение ролей даёт либо слабый поиск, либо мусор в выдаче.
6. **Включить всё сразу.** Пять приёмов одним коммитом — и непонятно, что помогло, а что навредило. По одному, с eval'ом после каждого — дисциплина урока 1.7.

## 🏋️ Упражнения

### Упражнение 1 — спасите лексическую кривую 🚑

Попробуйте «починить» акт 1: окно шире (`window=3`), стеммы короче (4 символа), фильтр стоп-слов. Стало ли можно ставить порог? Сформулируйте вывод: это предел параметров или предел подхода?

### Упражнение 2 — родитель на бюджете 💰

Модифицируйте `parent_search`: вместо целого родителя возвращайте **окно** — кусок родительского текста в ±300 символов вокруг найденного чанка (по границам слов). Сравните размер выдачи с полным родителем на обоих демо-вопросах.

### Упражнение 3 — LLM-реранкер на бумаге 📝

Напишите функцию `build_rerank_prompt(question, candidates) -> str`: системный промпт для LLM-судьи, который получает вопрос и нумерованные фрагменты, а вернуть должен строгий JSON с оценками релевантности 0–10 (все приёмы — из уроков 1.2 и 1.7: роль, разделители, формат, few-shot при желании). Вызывать LLM не нужно — промпт пригодится в модуле 3 в готовом виде.

## 📝 Мини-квиз

**1. Зачем приклеивать заголовок документа к чанку при индексации и почему не стоит показывать этот текст пользователю?**

<details><summary>Ответ</summary>

Глубокие чанки теряют контекст («Компенсация — отгул» — о чём это?), а вопросы часто содержат слова заголовка. Заголовок возвращает чанку тему почти бесплатно. Показывать же надо оригинал (обогащённый текст — техническая деталь индекса), поэтому исходник хранится отдельно в payload.

</details>

**2. Почему лексическая похожесть провалила поиск границ тем, а эмбеддинги справляются?**

<details><summary>Ответ</summary>

Соседние предложения одной темы часто не имеют общих слов — на коротких текстах пересечение лексики слишком редкое событие («пёс ≠ собака» из урока 2.1 в худшем виде). Эмбеддинги сравнивают смысл, а не буквы: «заявка на отпуск» и «неиспользованные дни» получают высокую близость, и настоящая граница тем становится единственной долиной кривой.

</details>

**3. В parent-document retrieval что оптимизируется мелким чанком, а что — родителем?**

<details><summary>Ответ</summary>

Мелкий чанк оптимизирует **поиск**: точный, сконцентрированный вектор — снайперское попадание. Родитель оптимизирует **ответ**: связный контекст для человека или LLM. Разделение ролей снимает компромисс «точность или контекст», оставляя только цену в токенах.

</details>

**4. Почему cross-encoder нельзя поставить на первую ступень поиска?**

<details><summary>Ответ</summary>

Он не производит векторов документов — нечего сложить в индекс. Каждая пара «запрос-документ» — полный проход модели: миллион документов × ~25 мс = часы на один запрос. Поэтому его место — вторая ступень, пересортировка 20–50 кандидатов от bi-encoder'а.

</details>

**5. В каком порядке внедрять приёмы модуля и почему?**

<details><summary>Ответ</summary>

От дешёвых к дорогим: структурный чанкинг → контекстные заголовки → метаданные/фильтры → и только по показаниям eval'а — parent-document, semantic chunking, cross-encoder. Каждый шаг отдельно, с замером: дешёвые приёмы часто закрывают проблему, не дав дорогим шанса потратить ваш бюджет.

</details>

## ✅ Решения упражнений

### Решение 1 — спасите лексическую кривую

In [ ]:
def stems_v2(sentence: str, stem_len: int = 4) -> set[str]:
    stop = {"это", "для", "или", "что", "как", "при", "все", "его", "них"}
    return {w[:stem_len] for w in tokenize(sentence)
            if len(w) > 2 and w not in stop}

def lexical_sim_v2(left: list[str], right: list[str]) -> float:
    a = set().union(*map(stems_v2, left)) if left else set()
    b = set().union(*map(stems_v2, right)) if right else set()
    return len(a & b) / len(a | b) if a | b else 0.0

curve_v2 = similarity_curve(sentences, lexical_sim_v2, window=3)
false_valleys = sum(1 for i, s in enumerate(curve_v2, 1)
                    if s == 0 and i not in true_boundaries)
print("Кривая v2 (окно 3, стеммы по 4, стоп-слова выброшены):\n")
for i, sim in enumerate(curve_v2, start=1):
    bar = "#" * int(sim * 50)
    mark = "  <-- настоящая граница" if i in true_boundaries else ""
    print(f"{i:3} {sim:.3f} {bar}{mark}")
print(f"\nНулевых 'долин' вне настоящих границ: {false_valleys}")

Кривая стала чуть глаже (широкое окно накапливает больше слов), но ложные долины никуда не делись, а настоящие границы по-прежнему не выделяются. **Это предел подхода, а не параметров**: сигнала «общие слова» на коротких отрезках просто недостаточно. Инженерный вывод — важный и общий: прежде чем тюнить параметры метода, спросите, есть ли в данных сигнал, который метод способен увидеть.

### Решение 2 — родитель на бюджете

In [ ]:
def parent_window(child: dict, radius: int = 300) -> str:
    """Кусок родителя в +-radius символов вокруг найденного чанка, по словам."""
    parent_text = parents[child["parent_id"]].text
    start = parent_text.find(child["text"][:50])   # где чанк лежит в родителе
    if start == -1:
        return child["text"]                        # не нашли (чанк склеен) - сам чанк
    left = max(0, start - radius)
    right = min(len(parent_text), start + len(child["text"]) + radius)
    # не режем слова по краям
    if left > 0:
        left = parent_text.find(" ", left) + 1
    if right < len(parent_text):
        right = parent_text.rfind(" ", left, right)
    return ("..." if left > 0 else "") + parent_text[left:right].strip() \
           + ("..." if right < len(parent_text) else "")

for question in ("Сколько компания доплачивает в первые дни болезни?",
                 "Какая надбавка за неделю дежурства?"):
    (score, child), *_ = parent_search(question, top_parents=1)
    full = parents[child["parent_id"]].text
    window = parent_window(child)
    print(f"Вопрос: {question}")
    print(f"  чанк {len(child['text'])} симв. -> окно {len(window)} симв. "
          f"(вместо родителя {len(full)} симв.)")
    print(f"  {window[:150]}...\n")

Три уровня бюджета на выбор: чанк (минимум токенов, куцый контекст) → окно (баланс) → родитель целиком (максимум связности). На наших коротких документах окно почти совпадает с родителем, но на реальных страницах в 10 000 символов разница — это деньги из урока 1.3, умноженные на каждый запрос.

### Решение 3 — LLM-реранкер на бумаге

In [ ]:
def build_rerank_prompt(question: str, candidates: list[str]) -> str:
    """Промпт LLM-судьи: все приёмы из уроков 1.2 и 1.7."""
    numbered = "\n\n".join(
        f"[{i}] {text}" for i, text in enumerate(candidates, start=1)
    )
    return f"""Ты — судья релевантности в поисковой системе.
Оцени, насколько каждый фрагмент отвечает на вопрос пользователя.

Шкала: 0 — не относится к вопросу, 5 — та же тема, но ответа нет,
10 — содержит прямой ответ.

Вопрос: {question}

Фрагменты:
<fragments>
{numbered}
</fragments>

Ответь строго JSON-объектом без пояснений, ключи — номера фрагментов:
{{"1": <0-10>, "2": <0-10>, ...}}"""

demo_candidates = [
    "Дежурство длится одну неделю. За неделю дежурства начисляется надбавка 15% к окладу.",
    "Отпуск можно дробить на части, но одна из частей должна быть не короче 14 дней.",
]
print(build_rerank_prompt("Какая надбавка за дежурство?", demo_candidates))

Все знакомые приёмы на месте: роль (урок 1.2), шкала с якорями (чтобы «7» значила одно и то же для разных фрагментов), данные в разделителях, строгий JSON-формат (валидировать его Pydantic'ом вы умеете с урока 1.2). В модуле 3 этот промпт поедет в Ollama и станет третьим реранкером в вашей коллекции — после диверсификации и cross-encoder'а.

## 🎓 Итоги

- **Контекстные заголовки** — приём за ноль рублей: индексируем «заголовок + чанк», показываем оригинал. Включён по умолчанию.
- **Semantic chunking** ищет границы тем по долинам кривой похожести. Лексика здесь слепа (вы видели шум своими глазами) — это единственный приём модуля, по-настоящему требующий эмбеддингов. Для текстов со структурой — избыточен.
- **Parent-document** разводит роли: мелкий чанк ищет, родитель отвечает. Бюджет контролируется окном.
- **Cross-encoder** читает пару «вопрос+документ» целиком — точнее bi-encoder'а, но без предвычислимых векторов: часы на полной базе, доли секунды на 30 кандидатах. Только вторая ступень.
- Карта приёмов: от бесплатных к дорогим, по одному, через eval.

**Дальше — урок 2.7, финал модуля:** сводный квиз и практика — семантический поиск по базе документации, собранный из всего арсенала модуля.

## 📚 Что почитать

- [Sentence-Transformers: Retrieve & Re-Rank](https://sbert.net/examples/applications/retrieve_rerank/README.html) — двухступенчатый поиск от авторов библиотеки
- [Карточка mmarco-mMiniLMv2](https://huggingface.co/cross-encoder/mmarco-mMiniLMv2-L12-H384-v1) — наш мультиязычный cross-encoder
- [Greg Kamradt: Levels of Text Splitting](https://github.com/FullStackRetrieval-com/RetrievalTutorials/blob/main/tutorials/LevelsOfTextSplitting/5_Levels_Of_Text_Splitting.ipynb) — известный разбор пяти уровней чанкинга, включая семантический
- [LangChain: ParentDocumentRetriever](https://python.langchain.com/docs/how_to/parent_document_retriever/) — тот же паттерн во фреймворке (вы уже написали его руками)
- [Anthropic: Contextual Retrieval](https://www.anthropic.com/news/contextual-retrieval) — развитие идеи контекстных заголовков: контекст чанка генерирует LLM